# Pooled surrogate-field specificity for direct Oλ
Use the observed dataset-adjusted arithmetic-mean matrices from the primary direct midpoint run. For each context in the current selection table, compare the learned continuous clinical direction with 10,000 coefficient permutations, sign randomizations, and norm-matched isotropic fields per family. Evaluate each surrogate on both regime matrices before computing the adverse-minus-favorable contrast. The adverse regime of direct Oλ is low.

The unconstrained tests use the saved standardized clinical coefficients. The compositional sensitivity applies the original draw-specific scale and CLR transformation to clinical and surrogate fields. A reduced geometry receives scale adjustment alone because removing coordinates breaks the full CLR sum constraint. All three metrics and their oriented joint statistic are reported.

Run all cells to process available bundles. Rerun the last cell to discover new or revised inputs. Optionally enable polling while the primary run is writing bundles. Set FINAL_SELECTION only after the primary selection table contains the complete intended comparator set. BH corrections are computed separately across contexts for each representation, analysis, family, and statistic.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import io
import json
import time
from zipfile import BadZipFile
import numpy as np
import pandas as pd
from IPython.display import display
drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
PRIMARY = DATA/'predictive_field_direct_midpoint_mean_dataset_adjusted_v1'
OUTPUT = DATA/'pooled_surrogate_fields_direct_midpoint_v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
WATCH_CURRENT_QUEUE = False
FINAL_SELECTION = False
COUNT = 10000
METRICS = ['prognostic_channeling','signed_graph_variation','signed_neighbor_cosine']
DESIRED_SIGN = np.array([1.,-1.,1.])
FAMILIES = ['permuted_field','sign_randomized_field','norm_matched_isotropic_field']
GEOMETRIES = {'TCAT_GEP':52,'atlas_cell_type':11}
COMPONENTS = {
    'TCAT_SCORE_ASA_CLR':{'TCAT_GEP_'+s+'_CLR' for s in ('OX40_EBI3','ICOS_CD38','CTLA4_CD38','TIMD4_TIM3')},
    'TCAT_SCORE_Proliferation_CLR':{'TCAT_GEP_CellCycle_'+s+'_CLR' for s in ('G2M','S','Late_S')},
}

In [ ]:
def safe_name(value):
    return ''.join(
        character if character.isalnum() or character in '._-' else '_'
        for character in str(value)
    )


def field_metric_matrices(correlation):
    correlation = np.asarray(correlation, dtype=float)
    first, second = np.triu_indices(len(correlation), 1)
    edge = correlation[first, second]
    weight = edge ** 2
    sign = np.where(edge < 0, -1.0, 1.0)

    degree = np.zeros(len(correlation), dtype=float)
    np.add.at(degree, first, weight)
    np.add.at(degree, second, weight)

    adjacency = np.zeros_like(correlation)
    adjacency[first, second] = weight * sign
    adjacency[second, first] = weight * sign
    laplacian = np.diag(degree) - adjacency
    return correlation, adjacency, laplacian, degree


def field_metrics_batch(correlation, fields):
    fields = np.atleast_2d(np.asarray(fields, dtype=float))
    correlation, adjacency, laplacian, degree = field_metric_matrices(correlation)

    field_norm = np.maximum(np.sum(fields ** 2, axis=1), 1e-12)
    channeling = np.sum((fields @ correlation) * fields, axis=1) / field_norm

    degree_energy = np.maximum(np.sum(fields ** 2 * degree[None, :], axis=1), 1e-12)
    variation = np.sum((fields @ laplacian) * fields, axis=1) / degree_energy

    neighbor = (fields @ adjacency) / np.maximum(degree[None, :], 1e-12)
    neighbor_energy = np.maximum(
        np.sum(neighbor ** 2 * degree[None, :], axis=1), 1e-12
    )
    cosine = (
        np.sum(fields * neighbor * degree[None, :], axis=1)
        / np.sqrt(degree_energy * neighbor_energy)
    )
    return np.column_stack([channeling, variation, cosine])


def mean_tangent_map(feature_sd_by_draw):
    feature_sd_by_draw = np.asarray(feature_sd_by_draw, dtype=float)
    dimensions = feature_sd_by_draw.shape[1]
    answer = np.zeros((dimensions, dimensions), dtype=float)
    identity = np.eye(dimensions)
    for scale in feature_sd_by_draw:
        if not np.isfinite(scale).all() or np.any(scale <= 0):
            raise ValueError('Feature-SD vectors must be positive and finite.')
        direction = scale / np.linalg.norm(scale)
        projection = identity - np.outer(direction, direction)
        answer += projection @ np.diag(scale)
    return answer / len(feature_sd_by_draw)


def canonical_raw_field(beta, coefficient_scales):
    beta = np.asarray(beta, dtype=float)
    coefficient_scales = np.asarray(coefficient_scales, dtype=float)
    if not np.isfinite(coefficient_scales).all() or np.any(coefficient_scales <= 0):
        raise ValueError('Coefficient feature scales are missing or invalid.')
    raw = beta / coefficient_scales
    raw -= raw.mean()
    if np.linalg.norm(raw) <= 1e-12:
        raise ValueError('The projected clinical field has zero norm.')
    return raw


def raw_field_without_constraint(beta, coefficient_scales):
    beta = np.asarray(beta, dtype=float)
    coefficient_scales = np.asarray(coefficient_scales, dtype=float)
    if not np.isfinite(coefficient_scales).all() or np.any(coefficient_scales <= 0):
        raise ValueError('Coefficient feature scales are missing or invalid.')
    raw = beta / coefficient_scales
    if np.linalg.norm(raw) <= 1e-12:
        raise ValueError('The clinical field has zero norm.')
    return raw


def mean_standardization_map(feature_sd_by_draw):
    feature_sd_by_draw = np.asarray(feature_sd_by_draw, dtype=float)
    if not np.isfinite(feature_sd_by_draw).all() or np.any(feature_sd_by_draw <= 0):
        raise ValueError('Feature-SD vectors must be positive and finite.')
    return np.diag(feature_sd_by_draw.mean(axis=0))


def normalize_rows(values, target_norm):
    values = np.asarray(values, dtype=float)
    norm = np.linalg.norm(values, axis=1)
    if np.any(norm <= 1e-12):
        raise ValueError('At least one surrogate has zero norm after projection.')
    return values * (target_norm / norm)[:, None]


def surrogate_matrix(base_field, family, rng, count, compositional):
    base_field = np.asarray(base_field, dtype=float)
    dimensions = len(base_field)
    target_norm = np.linalg.norm(base_field)

    if family == 'permuted_field':
        order = np.argsort(rng.random((count, dimensions)), axis=1)
        values = base_field[order]
    elif family == 'sign_randomized_field':
        values = np.abs(base_field)[None, :] * rng.choice(
            [-1.0, 1.0], size=(count, dimensions)
        )
    elif family == 'norm_matched_isotropic_field':
        values = rng.normal(size=(count, dimensions))
    else:
        raise ValueError(family)

    if compositional:
        values -= values.mean(axis=1, keepdims=True)
    return normalize_rows(values, target_norm)


def surrogate_summary(observed, null):
    null = np.asarray(null, dtype=float)
    null_mean = null.mean(axis=0)
    null_sd = np.maximum(null.std(axis=0, ddof=1), 1e-12)
    z = (observed - null_mean) / null_sd
    p = (
        1 + np.sum(null * DESIRED_SIGN >= observed * DESIRED_SIGN, axis=0)
    ) / (len(null) + 1)

    oriented_z = z * DESIRED_SIGN
    joint_z = oriented_z.mean()
    joint_null = ((null - null_mean) / null_sd * DESIRED_SIGN).mean(axis=1)
    joint_p = (1 + np.sum(joint_null >= joint_z)) / (len(null) + 1)
    return z, p, joint_z, joint_p, joint_null


def benjamini_hochberg(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values)
    adjusted = values[order] * len(values) / np.arange(1, len(values) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    answer = np.empty(len(values), dtype=float)
    answer[order] = np.minimum(adjusted, 1)
    return answer

In [ ]:
def projection_diagnostic(scale_by_draw, raw_field):
    residuals = []
    fractions_removed = []
    directions = []
    for scale in np.asarray(scale_by_draw, dtype=float):
        direction = scale / np.linalg.norm(scale)
        unprojected = scale * raw_field
        projected = unprojected - direction * (direction @ unprojected)
        residuals.append(
            abs(direction @ projected) / max(np.linalg.norm(projected), 1e-12)
        )
        fractions_removed.append(
            np.linalg.norm(unprojected - projected)
            / max(np.linalg.norm(unprojected), 1e-12)
        )
        directions.append(direction)
    directions = np.asarray(directions)
    representative = directions.mean(axis=0)
    representative /= np.linalg.norm(representative)
    cosines = directions @ representative
    return {
        'maximum_constraint_residual': float(np.max(residuals)),
        'median_fraction_field_norm_removed': float(np.median(fractions_removed)),
        'maximum_fraction_field_norm_removed': float(np.max(fractions_removed)),
        'minimum_constraint_direction_cosine': float(np.min(cosines)),
        'q025_constraint_direction_cosine': float(np.quantile(cosines, 0.025)),
    }



def evaluate(a, field, context, destination):
    meta=json.loads(str(a['metadata_json']))
    if (meta['representation'],meta['context'],meta['scope'],meta['geometry']) != (field,context,'pooled','dataset_adjusted'):
        raise ValueError(f'Unexpected metadata for {field} / {context}')
    assert meta['aggregation']=='arithmetic mean of draw-specific correlation matrices'
    names=a['feature_names'].astype(str)
    removed=set(a['removed_coordinates'].astype(str))
    same=(field=='TCAT_GEP' and context.startswith('TCAT_GEP_')) or (field=='atlas_cell_type' and context.startswith('ATLAS_proportion_'))
    expected={context} if same else COMPONENTS.get(context,set()) if field=='TCAT_GEP' else set()
    assert removed==expected and removed.isdisjoint(names) and len(set(names))==len(names)==GEOMETRIES[field]-len(expected)
    adverse_high=bool(a['adverse_high_on_original_predictor'])
    if context=='O_lambda' and adverse_high: raise ValueError('Direct Oλ must use low as adverse.')
    q=a['raw_probability_high']
    assert len(q)==163 and np.isfinite(q).all() and ((q>=0)&(q<=1)).all()
    np.testing.assert_allclose(a['adverse_regime_membership'],q if adverse_high else 1-q,atol=1e-12)
    matrices=[a[k+'_mean_correlation'] for k in ('adverse','favorable')]
    sd=[a[k+'_feature_sd_by_draw'].astype(float) for k in ('adverse','favorable')]
    for R,S in zip(matrices,sd):
        assert R.shape==(len(names),len(names)) and S.shape==(500,len(names))
        assert np.isfinite(S).all() and (S>0).all()
        np.testing.assert_allclose(R,R.T,rtol=0,atol=1e-10)
        np.testing.assert_allclose(np.diag(R),1,rtol=0,atol=1e-10)
        if np.linalg.eigvalsh(R)[0]<-1e-10: raise ValueError('Observed matrix failed PSD check.')
    beta=a['clinical_beta'].astype(float)
    assert beta.shape==(len(names),) and np.isfinite(beta).all() and np.linalg.norm(beta)>0
    scale=a['coefficient_feature_scales']
    rows=[]; diagnostics=[]; arrays={}
    for analysis in ['unconstrained','compositional_sensitivity']:
        project=analysis=='compositional_sensitivity' and not removed
        if analysis=='unconstrained':
            base=beta
            maps=[np.eye(len(beta))]*2
        elif project:
            base=canonical_raw_field(beta,scale)
            maps=[mean_tangent_map(s) for s in sd]
            for regime,s in zip(('adverse','favorable'),sd):
                diagnostics.append(dict(regime=regime,**projection_diagnostic(s,base)))
        else:
            base=raw_field_without_constraint(beta,scale)
            maps=[mean_standardization_map(s) for s in sd]
        def contrast(fields):
            return (field_metrics_batch(matrices[0],fields @ maps[0].T)
                    - field_metrics_batch(matrices[1],fields @ maps[1].T))
        obs=contrast(base[None,:])[0]
        for family in FAMILIES:
            # Pair the underlying random numbers across both sensitivity analyses.
            seed=int.from_bytes(hashlib.sha256(f'{field}/{context}/{family}/0'.encode()).digest()[:8],'little')
            fields=surrogate_matrix(base,family,np.random.default_rng(seed),COUNT,project)
            null=contrast(fields)
            if not np.isfinite(null).all() or not np.isfinite(obs).all(): raise ValueError('Nonfinite contrast')
            z,p,jz,jp,jn=surrogate_summary(obs,null)
            prefix=analysis+'__'+family
            arrays[prefix+'__null']=null
            arrays[prefix+'__joint_null']=jn
            row=dict(representation=field,context=context,analysis=analysis,surrogate_family=family,
                     seed=str(seed),surrogates=COUNT,nodes=len(names),removed_coordinates=';'.join(sorted(removed)),
                     clinical_adverse_tail='high' if adverse_high else 'low',constraint_applied=bool(project),
                     joint_oriented_z=float(jz),joint_surrogate_p=float(jp))
            for j,metric in enumerate(METRICS):
                row['contrast__'+metric]=float(obs[j])
                row['z__'+metric]=float(z[j])
                row['p__'+metric]=float(p[j])
            rows.append(row)
    tmp=destination/'nulls.tmp.npz'
    np.savez_compressed(tmp,**arrays)
    tmp.replace(destination/'nulls.npz')
    return rows,diagnostics

In [ ]:
def csv_atomic(frame,name):
    tmp=OUTPUT/(name+'.tmp')
    frame.to_csv(tmp,index=False)
    tmp.replace(OUTPUT/name)

def scan():
    queue_path=PRIMARY/'selected_contexts.csv'
    if not queue_path.exists():
        print('Waiting for the primary selection table:',queue_path)
        return False
    queue=pd.read_csv(queue_path)
    assert queue.context.is_unique and 'O_lambda' in set(queue.context)
    rows=[]; diagnostics=[]; inventory=[]
    for entry in queue.itertuples(index=False):
        for field in GEOMETRIES:
            path=PRIMARY/'compositional_sensitivity_bundles'/field/(safe_name(entry.context)+'.npz')
            item=dict(representation=field,context=entry.context,selection=entry.selection)
            inventory.append(item)
            if entry.status!='ready' or not path.exists():
                item['status']=entry.status if entry.status!='ready' else 'awaiting observed bundle'
                continue
            # Read an immutable snapshot; the producer may still be compressing this file.
            payload=path.read_bytes()
            fingerprint=hashlib.sha256(payload).hexdigest()
            dest=OUTPUT/'checkpoints'/field/safe_name(entry.context)/f'v1_{COUNT}_{fingerprint}'
            dest.mkdir(parents=True,exist_ok=True)
            checkpoint=dest/'results.json'
            if checkpoint.exists() and (dest/'nulls.npz').exists():
                saved=json.loads(checkpoint.read_text())
            else:
                try:
                    with np.load(io.BytesIO(payload),allow_pickle=False) as a:
                        batch,diag=evaluate(a,field,entry.context,dest)
                except (BadZipFile,EOFError):
                    item['status']='bundle incomplete; retry on next scan'
                    continue
                except ValueError as error:
                    if not payload or not payload.startswith(b'PK'):
                        item['status']='bundle unreadable; retry on next scan'
                        continue
                    raise
                saved=dict(source=str(path),sha256=fingerprint,results=batch,diagnostics=diag)
                tmp=checkpoint.with_suffix('.tmp')
                tmp.write_text(json.dumps(saved,indent=2)); tmp.replace(checkpoint)
                print('Saved',field,entry.context,flush=True)
            rows.extend(dict(record,selection=entry.selection) for record in saved['results'])
            diagnostics.extend(dict(record,representation=field,context=entry.context) for record in saved['diagnostics'])
            item.update(status='complete',checkpoint=str(checkpoint))
    inventory=pd.DataFrame(inventory)
    complete=inventory.status.eq('complete').all()
    status='final' if FINAL_SELECTION and complete else 'interim'
    csv_atomic(inventory,'bundle_status.csv')
    results=pd.DataFrame(rows)
    if rows:
        for _,group in results.groupby(['representation','analysis','surrogate_family']):
            for source,target in [('joint_surrogate_p','joint_FDR')]+[('p__'+m,'FDR__'+m) for m in METRICS]:
                results.loc[group.index,target]=benjamini_hochberg(group[source])
        results['adjustment_status']=status
        csv_atomic(results,'pooled_surrogate_results.csv')
        csv_atomic(pd.DataFrame(diagnostics),'projection_diagnostics.csv')
        display(results.loc[results.context.eq('O_lambda'),['representation','analysis','surrogate_family',
                         'joint_oriented_z','joint_surrogate_p','joint_FDR','adjustment_status']])
    else:
        csv_atomic(results,'pooled_surrogate_results.csv')
        csv_atomic(pd.DataFrame(),'projection_diagnostics.csv')
    print(f'{inventory.status.eq("complete").sum()}/{len(inventory)} pairs complete | BH status: {status}',flush=True)
    return complete

while True:
    if scan() or not WATCH_CURRENT_QUEUE:
        break
    time.sleep(60)
print('Output:',OUTPUT)